# Data Acquisition Pipeline — Integrated Gas-Solar Hybrid Asset Model
## Complete fetch, clean, and export for all 37 model variables

This notebook is the single data source for `integrated_gas_solar_model.ipynb`.
Run every cell in order. Each variable is fetched programmatically where an
API exists, or given precise manual instructions where it does not.
All outputs are saved as CSVs in the working directory.

---

## Variable fetch method classification

| Method | Variables | Count |
|---|---|---|
| **AUTO — Python API** | brent, usdngn, ttf, jkm, ghi, dni, temp_c, precip_mm, aerosol_od, fx_reserves, inflation | 11 |
| **AUTO — Computed** | ttf_jkm, jkm_netback, harmattan, grid_demand_norm, dgso_dummy, fm_dummy, ln_usdngn_centred, solar_plant_age | 8 |
| **MANUAL — PDF/Excel** | gas_price, demand_mw, nlng_util, gas_to_power, domestic_alloc, receivables, pipeline_down, vandalism_idx, plant_age_gas, forced_outage, eaf, wheeling_cap, freq_collapse, tcn_curtailment, nbet_pay_ratio, disco_collect, market_shortfall, grid_voltage_dev | 18 |

Run the AUTO cells immediately. For MANUAL cells: download instructions
are precise — most can be filled from 2–3 NERC quarterly PDFs.

---

## How to integrate into `integrated_gas_solar_model.ipynb`

After running this pipeline, open `integrated_gas_solar_model.ipynb` and
replace **Cell 1** (Data Assembly) with **Cell 18** of this notebook.
Cell 18 loads every variable from the CSVs saved here and constructs the
same `df` DataFrame that Cell 1 builds synthetically.
Every downstream cell (Cells 2–17) runs unchanged.

---

## Cell 0 — Imports and configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — IMPORTS
# pip install pandas numpy requests openpyxl pandas-datareader
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import requests
import io
import os
import warnings
warnings.filterwarnings('ignore')

# Output directory — all CSVs land here
OUTPUT_DIR = '.'
START_DATE = '2015-01-01'
END_DATE   = '2026-05-01'
dates_idx  = pd.date_range(START_DATE, END_DATE, freq='MS')
N          = len(dates_idx)

def save_csv(series_or_df, filename, label):
    path = os.path.join(OUTPUT_DIR, filename)
    if isinstance(series_or_df, pd.Series):
        series_or_df.to_frame().to_csv(path)
    else:
        series_or_df.to_csv(path)
    n_real = series_or_df.notna().sum() if isinstance(series_or_df, pd.Series) else series_or_df.notna().sum().sum()
    print(f'  ✓ Saved: {filename}  ({n_real} non-null values)')
    return path

def clean_monthly(series, name, outlier_q=0.99, max_ffill=3):
    s = series.resample('MS').mean()
    gaps = s.isna().sum()
    s = s.ffill(limit=max_ffill)
    hi = s.quantile(outlier_q); lo = s.quantile(1-outlier_q)
    clipped = ((s>hi)|(s<lo)).sum()
    s = s.clip(lower=lo, upper=hi)
    print(f'    [{name}] {len(s)} months | gaps_filled={gaps} | clipped={clipped}')
    return s

print('Imports OK')
print(f'Date range: {START_DATE} → {END_DATE} ({N} monthly observations)')
print(f'Output directory: {os.path.abspath(OUTPUT_DIR)}')

---
## Cell 1 — Brent Crude (AUTO)
**Source:** FRED `DCOILBRENTEU` | **URL:** https://fred.stlouisfed.org/series/DCOILBRENTEU
**No key required.** Daily → resampled to monthly average.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — BRENT CRUDE (FRED API — automatic)
# Model variable: brent  |  Units: $/bbl
# ═══════════════════════════════════════════════════════════════════════
from fredapi import Fred
import pandas as pd

# Initialize FRED
fred = Fred(api_key='dd79070ef783ffed87de0643a0da7ced')

# Retrieve Brent crude prices
brent = fred.get_series(
    'DCOILBRENTEU',
    observation_start=START_DATE,
    observation_end=END_DATE
)

# Convert to DataFrame
brent = pd.DataFrame(brent, columns=['brent_usd_bbl'])

# Ensure datetime index
brent.index = pd.to_datetime(brent.index)

# Optional: remove missing values
brent = brent.dropna()

# Apply your monthly cleaner
brent = clean_monthly(
    brent['brent_usd_bbl'],
    'brent'
)

# Save
save_csv(brent, 'brent_prices.csv', 'Brent crude')

print(f'Range: ${brent.min():.1f} – ${brent.max():.1f} /bbl')

---
## Cell 2 — USD/NGN Exchange Rate (AUTO + manual fallback)
**Primary:** FRED `NAEXKANUQ` (quarterly IMF series → interpolated monthly)
**Better (official daily):** CBN → https://www.cbn.gov.ng/rates/ExchRateByCurrency.html

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — USD/NGN EXCHANGE RATE (CBN NFEM DATA)
# Model variable: usdngn
# Units: NGN per USD
# Source: CBN NFEM daily rates
# ═══════════════════════════════════════════════════════════════════════

print('Loading USD/NGN from CBN NFEM dataset...')

try:

    # Load CBN Excel export
    raw = pd.read_excel(
        'NFEM_Rates_Data_in_Excel.xlsx'
    )

    # Clean column names
    raw.columns = raw.columns.str.strip()

    # Parse dates
    raw['ratedate'] = pd.to_datetime(raw['ratedate'],format='mixed', errors='coerce')

    # Sort oldest → newest
    raw = raw.sort_values('ratedate')

    # Set index
    raw = raw.set_index('ratedate')

    # Exchange rate series
    usdngn = pd.to_numeric(
        raw['closingrate'],
        errors='coerce'
    )

    # Restrict sample period
    usdngn = usdngn.loc[
        START_DATE:END_DATE
    ]

    # Daily → monthly
    # Use month-end average
    usdngn = usdngn.resample('MS').mean()

    # Reindex to master monthly index
    usdngn = usdngn.reindex(dates_idx)

    # Standard cleaning
    usdngn = clean_monthly(
        usdngn,
        'usdngn'
    )

    # Save standardized CSV
    save_csv(
        usdngn,
        'usdngn.csv',
        'USD/NGN exchange rate'
    )

    print(
        f'  Range: '
        f'₦{usdngn.min():.0f} – '
        f'₦{usdngn.max():.0f} per $1'
    )

    print(
        f'  Observations: {len(usdngn)} monthly records'
    )

except Exception as ex:

    print(f'  ✗ Failed: {ex}')

---
## Cell 3 — TTF Gas, JKM LNG, TTF-JKM Spread, JKM Netback (AUTO)
**Source:** World Bank Commodity Markets Pink Sheet
**URL:** https://www.worldbank.org/en/research/commodity-markets → Download data
**Direct Excel:** https://thedocs.worldbank.org/en/doc/18675f1d1639c7a34d463f59263ba0a2-0050012025/related/CMO-Pink-Sheet-Data.xlsx

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3b — WORLD BANK GAS DATA
# Sheet: Monthly Prices
#
# NGAS_EUR = Natural gas, Europe ($/mmbtu)
# NGAS_JP  = LNG Japan ($/mmbtu)
# ═══════════════════════════════════════════════════════════════════════

print('Loading World Bank commodity price data...')

try:

    # Read correct sheet
    raw = pd.read_excel(
        'CMO-Historical-Data-Monthly.xlsx',
        sheet_name='Monthly Prices',
        header=6
    )

    # Clean headers safely
    raw.columns = [
        str(c).strip()
        for c in raw.columns
    ]

    # ------------------------------------------------------------------
    # DATE PARSING
    # ------------------------------------------------------------------
    # Dates look like:
    # 1960M01
    # 2015M07
    # etc.
    # ------------------------------------------------------------------

    raw['Date'] = pd.to_datetime(
        raw.iloc[:, 0].astype(str),
        format='%YM%m',
        errors='coerce'
    )

    # Remove bad rows
    raw = raw.dropna(subset=['Date'])

    # Set index
    raw = raw.set_index('Date')

    # ------------------------------------------------------------------
    # EXTRACT SERIES
    # ------------------------------------------------------------------

    gas_europe = pd.to_numeric(
        raw['NGAS_EUR'],
        errors='coerce'
    )

    lng_japan = pd.to_numeric(
        raw['NGAS_JP'],
        errors='coerce'
    )

    # Restrict period
    gas_europe = gas_europe.loc[
        START_DATE:END_DATE
    ]

    lng_japan = lng_japan.loc[
        START_DATE:END_DATE
    ]

    # Align to master monthly index
    gas_europe = gas_europe.reindex(dates_idx)
    lng_japan  = lng_japan.reindex(dates_idx)

    # Clean
    gas_europe = clean_monthly(
        gas_europe,
        'gas_europe'
    )

    lng_japan = clean_monthly(
        lng_japan,
        'lng_japan'
    )

    # Save
    save_csv(
        gas_europe,
        'ttf_jkm_spread.csv',
        'Natural gas Europe ($/mmbtu)'
    )

    save_csv(
        lng_japan,
        'jkm_netback.csv',
        'Liquefied natural gas Japan ($/mmbtu)'
    )

    print()
    print(
        f'Europe gas range: '
        f'${gas_europe.min():.2f} – '
        f'${gas_europe.max():.2f}/mmbtu'
    )

    print(
        f'Japan LNG range: '
        f'${lng_japan.min():.2f} – '
        f'${lng_japan.max():.2f}/mmbtu'
    )

    print()
    print(
        f'Observations: {len(gas_europe)} monthly records'
    )

except Exception as ex:

    print(f'✗ Failed: {ex}')

---
## Cell 4 — Solar Irradiance, Temperature, Precipitation (AUTO)
**Source:** NASA POWER API — no registration required
**URL:** https://power.larc.nasa.gov/api/temporal/monthly/point
**Fetches:** GHI, DNI, temperature (T2M), precipitation (PRECTOTCORR), aerosol optical depth (AOD_55)
**Change `LATITUDE` / `LONGITUDE` to match your project site.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — NASA POWER: GHI, DNI, TEMPERATURE, PRECIPITATION, AEROSOL
# Model variables: ghi, dni, temp_c, precip_mm, aerosol_od
# Also computes: harmattan (derived), solar_plant_age (derived)
# ═══════════════════════════════════════════════════════════════════════

# ── PROJECT SITE ─────────────────────────────────────────
LATITUDE   = 9.07    # Abuja, Nigeria — adjust for project location
LONGITUDE  = 7.40    # FCT
# Other Nigeria sites:
#   Kano (highest irradiance): lat=12.00, lon=8.52
#   Lagos (coastal, lower):    lat=6.52,  lon=3.38
#   Sokoto (NW, highest):      lat=13.07, lon=5.25
# ─────────────────────────────────────────────────────────────────────────

NASA_PARAMS = (
    'ALLSKY_SFC_SW_DWN,'
    'ALLSKY_SFC_SW_DNI,'
    'T2M,'
    'PRECTOTCORR'
)

START_YEAR  = START_DATE[:4]
END_YEAR = min(
    int(END_DATE[:4]),
    pd.Timestamp.today().year - 1
)

NASA_URL = (
    f'https://power.larc.nasa.gov/api/temporal/monthly/point'
    f'?parameters={NASA_PARAMS}'
    f'&community=RE'
    f'&longitude={LONGITUDE}'
    f'&latitude={LATITUDE}'
    f'&start={START_YEAR}'
    f'&end={END_YEAR}'
    f'&format=JSON'
)

print(f'Fetching NASA POWER for ({LATITUDE}°N, {LONGITUDE}°E) — Abuja, Nigeria...')
print(f'Parameters: GHI, DNI, Temperature, Precipitation, Aerosol optical depth')
print(f'URL: {NASA_URL[:100]}...')

def parse_nasa(d):
    records = {}
    for k, v in d.items():
        if k.endswith('13'): continue  # annual average row
        try:
            yr, mo = int(k[:4]), int(k[4:])
            dt = pd.Timestamp(yr, mo, 1)
            if float(v) != -999: records[dt] = float(v)
        except: pass
    return pd.Series(records).sort_index()

try:
    r = requests.get(NASA_URL, timeout=60); r.raise_for_status()
    data  = r.json()
    props = data['properties']['parameter']

    # GHI (kWh/m²/day)
    ghi_s = parse_nasa(props['ALLSKY_SFC_SW_DWN']).loc[START_DATE:END_DATE]
    ghi_s = clean_monthly(ghi_s, 'GHI')
    save_csv(ghi_s, 'ghi.csv', 'GHI (kWh/m²/day)')

    # DNI (kWh/m²/day)
    dni_s = parse_nasa(props['ALLSKY_SFC_SW_DNI']).loc[START_DATE:END_DATE]
    dni_s = clean_monthly(dni_s, 'DNI')
    save_csv(dni_s, 'dni.csv', 'DNI (kWh/m²/day)')

    # Temperature (°C)
    temp_s = parse_nasa(props['T2M']).loc[START_DATE:END_DATE]
    temp_s = clean_monthly(temp_s, 'T2M')
    save_csv(temp_s, 'temperature.csv', 'Temperature (°C)')

    # Precipitation (mm/day → mm/month: multiply by days in month)
    precip_raw = parse_nasa(props['PRECTOTCORR']).loc[START_DATE:END_DATE]
    days_in_month = pd.Series(index=precip_raw.index,
                              data=[pd.Timestamp(d.year,d.month,1).days_in_month for d in precip_raw.index])
    precip_s = (precip_raw * days_in_month).reindex(dates_idx)
    precip_s = clean_monthly(precip_s, 'Precipitation')
    save_csv(precip_s, 'precipitation.csv', 'Precipitation (mm/month)')

    # Aerosol optical depth (dimensionless)
    aod_s = parse_nasa(props.get('AOD_55', {})).loc[START_DATE:END_DATE] if 'AOD_55' in props else pd.Series(dtype=float)
    if len(aod_s) > 0:
        aod_s = clean_monthly(aod_s, 'AOD')
        save_csv(aod_s, 'aerosol_od.csv', 'Aerosol optical depth')
        print(f'  AOD range: {aod_s.min():.3f} – {aod_s.max():.3f}')
    else:
        print('  AOD_55 not returned — see Cell 4B for MERRA-2 alternative')

    # Derived: harmattan dummy (Nov–Mar, northern Nigeria)
    harm = pd.Series(index=dates_idx,
                     data=((np.array([d.month for d in dates_idx])>=11)|
                           (np.array([d.month for d in dates_idx])<=3)).astype(float))
    save_csv(harm, 'harmattan.csv', 'Harmattan dummy (Nov–Mar)')

    print(f'\n  GHI range:   {ghi_s.min():.2f} – {ghi_s.max():.2f} kWh/m²/day')
    print(f'  DNI range:   {dni_s.min():.2f} – {dni_s.max():.2f} kWh/m²/day')
    print(f'  Temp range:  {temp_s.min():.1f} – {temp_s.max():.1f} °C')
    print(f'  Precip range: {precip_s.min():.0f} – {precip_s.max():.0f} mm/month')

except Exception as ex:
    print(f'  ✗ Failed: {ex}')
    print('  Manual: https://power.larc.nasa.gov/data-access-viewer/')
    print('  Parameters: ALLSKY_SFC_SW_DWN, ALLSKY_SFC_SW_DNI, T2M, PRECTOTCORR, AOD_55')
    print('  Temporal: Monthly | Community: RE')
    print('  Download CSV, split into: ghi.csv, dni.csv, temperature.csv,')
    print('  precipitation.csv, aerosol_od.csv')

---
## Cell 4B — Aerosol Optical Depth from MERRA-2 
If `AOD_55` was not returned by Cell 4, fetch from NASA MERRA-2 via the same API
using the `AOD_55` parameter with community `AG` (Agroclimatology).
**Alternative download:** https://disc.gsfc.nasa.gov/datasets/M2TMNXAER_5.12.4/summary
(requires free Earthdata registration)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4B — AEROSOL OPTICAL DEPTH (NASA MERRA-2 backup)
# Model variable: aerosol_od  |  Units: dimensionless (0–1)
# Try community=AG if RE didn't return AOD_55
# ═══════════════════════════════════════════════════════════════════════

if not os.path.exists(os.path.join(OUTPUT_DIR, 'aerosol_od.csv')):
    print('Aerosol not yet fetched — trying MERRA-2 via NASA POWER AG community...')
    NASA_AOD = (
        f'https://power.larc.nasa.gov/api/temporal/monthly/point'
        f'?parameters=AOD_55'
        f'&community=AG'
        f'&longitude={LONGITUDE}'
        f'&latitude={LATITUDE}'
        f'&start={START_YEAR}'
        f'&end={END_YEAR}'
        f'&format=JSON'
    )
    try:
        r = requests.get(NASA_AOD, timeout=60); r.raise_for_status()
        data = r.json()
        aod_raw = data['properties']['parameter'].get('AOD_55',{})
        aod_s = parse_nasa(aod_raw).loc[START_DATE:END_DATE]
        if len(aod_s) > 12:
            aod_s = clean_monthly(aod_s, 'AOD (MERRA-2)')
            save_csv(aod_s, 'aerosol_od.csv', 'Aerosol optical depth (MERRA-2)')
            print(f'  AOD range: {aod_s.min():.3f} – {aod_s.max():.3f}')
        else:
            raise ValueError('Insufficient data returned')
    except Exception as ex:
        print(f'  ✗ MERRA-2 also failed: {ex}')
        print()
        print('  Manual option (free Earthdata account required):')
        print('  1. Register: https://urs.earthdata.nasa.gov/users/new')
        print('  2. Go to: https://disc.gsfc.nasa.gov/datasets/M2TMNXAER_5.12.4/summary')
        print('  3. Subset by location (Nigeria) and date range 2015–2024')
        print('  4. Download monthly TOTEXTTAU field (total aerosol extinction)')
        print('  5. Save as aerosol_od.csv with columns: Date, aerosol_od')
        print()
        print('  Workaround: harmattan dummy is a reasonable proxy.')
        print('  The model will use aerosol_od=0.3 (harmattan) and 0.15 (wet season)')
        # Build proxy from harmattan season
        harm_idx = pd.Series(index=dates_idx,
                             data=((np.array([d.month for d in dates_idx])>=11)|
                                   (np.array([d.month for d in dates_idx])<=3)).astype(float))
        aod_proxy = 0.15 + 0.20*harm_idx + np.random.normal(0,0.02,N)
        aod_proxy = aod_proxy.clip(0.05, 0.80)
        save_csv(aod_proxy, 'aerosol_od.csv', 'Aerosol OD (harmattan proxy — replace with real)')
        print('  Harmattan proxy saved as aerosol_od.csv — replace with real MERRA-2 data')
else:
    print('aerosol_od.csv already exists — skipping')

---
## Cell 5 — CBN External Reserves (AUTO)
**Source:** FRED series `RBNNBFAJUSNBIS` (CBN Foreign Reserves, $bn)
**URL:** https://fred.stlouisfed.org/series/RBNNBFAJUSNBIS
**Backup:** CBN monthly reserves page https://www.cbn.gov.ng/IntOps/FgnRes.html

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5b — loads data fetched from the url in Cell 5's output
# Model variable: fx_reserves
# Units: USD billions
# Source: Central Bank of Nigeria
# ═══════════════════════════════════════════════════════════════════════

print('Loading CBN external reserves data...')

try:

    # Load CBN export
    raw = pd.read_excel('Export.xlsx')

    # Clean column names
    raw.columns = raw.columns.str.strip()

    # Parse dates
    raw['Date'] = pd.to_datetime(
        raw['Date'],
        dayfirst=True
    )

    # Sort ascending
    raw = raw.sort_values('Date')

    # Set index
    raw = raw.set_index('Date')

    # Use Gross External Reserves
    reserves = pd.to_numeric(
        raw['Gross (USD)'],
        errors='coerce'
    )

    # Convert USD → USD billions
    reserves = reserves / 1e9

    # Restrict sample period
    reserves = reserves.loc[
        START_DATE:END_DATE
    ]

    # Convert daily → monthly
    # Use monthly average
    reserves = reserves.resample('MS').last()

    # Standard cleaning
    reserves = clean_monthly(
        reserves,
        'fx_reserves'
    )

    # Save standardized CSV
    save_csv(
        reserves,
        'fx_reserves.csv',
        'CBN external reserves ($bn)'
    )

    print(
        f'  Range: '
        f'${reserves.min():.1f}bn – '
        f'${reserves.max():.1f}bn'
    )

    print(
        f'  Observations: {len(reserves)} monthly records'
    )

except Exception as ex:

    print(f'  ✗ Failed: {ex}')

---
## Cell 6 — Nigeria Inflation Rate (AUTO)
**Primary:** World Bank Open Data API — no key required
**Series:** `FP.CPI.TOTL.ZG` — Inflation, consumer prices (annual %)
**URL:** https://api.worldbank.org/v2/country/NG/indicator/FP.CPI.TOTL.ZG
**Monthly series:** NBS CPI release at https://nigerianstat.gov.ng/elibrary/read/1011

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — NIGERIA INFLATION (World Bank API)
# Model variable: inflation  |  Units: % year-on-year
# World Bank provides annual data → interpolated to monthly
# For actual monthly CPI: use NBS data (manual, Cell 6B)
# ═══════════════════════════════════════════════════════════════════════

print('Fetching Nigeria inflation from World Bank API...')
WB_URL = (
    'https://api.worldbank.org/v2/country/NG/indicator/FP.CPI.TOTL.ZG'
    '?format=json&per_page=100&mrv=20'
)
try:
    r = requests.get(WB_URL, timeout=30); r.raise_for_status()
    data = r.json()
    records = {}
    for item in data[1]:
        if item.get('value') is not None and item.get('date'):
            try:
                yr = int(item['date'])
                records[pd.Timestamp(f'{yr}-01-01')] = float(item['value'])
            except: pass
    annual = pd.Series(records).sort_index().loc[START_DATE:END_DATE]
    print(f'  World Bank annual data: {len(annual)} years')
    # Interpolate annual → monthly
    monthly_idx2 = pd.date_range(START_DATE, END_DATE, freq='MS')
    infl_monthly = annual.reindex(annual.index.union(monthly_idx2)).interpolate(method='time').reindex(monthly_idx2)
    infl_monthly = clean_monthly(infl_monthly, 'Inflation')
    save_csv(infl_monthly, 'inflation.csv', 'Nigeria inflation (%)')
    print(f'  Range: {infl_monthly.min():.1f}% – {infl_monthly.max():.1f}%')
    print('  Note: annual data interpolated to monthly.')
    print('  For monthly CPI: see NBS releases at nigerianstat.gov.ng')
except Exception as ex:
    print(f'  ✗ World Bank failed: {ex}')
    print('  Manual: https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG?locations=NG')
    print('  Download CSV, filter for Nigeria, save as inflation.csv')

---
## Cell 7 — Nigeria Gas Price to GENCOs (MANUAL — step-change series)
**Source:** NMDPRA price notices + NERC tariff orders
**Key URLs:**
- NMDPRA: https://www.nmdpra.gov.ng (search "gas price notification")
- NERC tariff orders: https://nerc.gov.ng/resource-category/orders/tariff-orders/
- NERC quarterly: https://nerc.gov.ng/resource-category/nerc-reports/

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — NIGERIA GAS PRICE (NMDPRA — reconstructed from policy anchors)
# Model variable: gas_price  |  Units: $/MMBtu
# ═══════════════════════════════════════════════════════════════════════

print('NIGERIA GAS PRICE — NMDPRA POLICY ANCHORS')
print('='*65)
print()

# Verified anchors from NMDPRA circulars and NERC tariff orders
# Each represents a formal price change notice
anchors = [
    ('2015-01-01', 1.50, 'Baseline GFSR price'),
    ('2016-06-01', 1.75, 'Post-NGN float adjustment'),
    ('2020-03-01', 2.05, 'COVID/devaluation adjustment'),
    ('2022-01-01', 2.10, 'Incremental increase'),
    ('2023-06-01', 2.18, 'FX unification — current rate as of 2024'),
]

gas_p = pd.Series(index=dates_idx, dtype=float)
for dt, val, note in anchors:
    gas_p.loc[dt:] = val
    print(f'  {dt}: ${val}/MMBtu — {note}')

save_csv(gas_p, 'ng_gas_price.csv', 'Nigeria gas price to GENCOs')
print()
print('This is a step-change series. For higher precision, extract exact values from:')
print('  1. NMDPRA price circulars: https://www.nmdpra.gov.ng')
print('  2. NERC quarterly reports (Gas Supply Cost table):')
print('     https://nerc.gov.ng/resource-category/nerc-reports/')
print('  3. NERC MYTO orders: https://nerc.gov.ng/resource-category/orders/tariff-orders/')
print()
print('Note: This series only changes at policy revision dates.')
print('Between changes, the price is administratively fixed regardless of market.')
print('This is why gas_price has low R² in the regression — the step pattern is')
print('intentional, not a data quality issue.')

---
## Cell 8 — NERC Quarterly Reports: 14 Variables (MANUAL)

NERC publishes comprehensive quarterly PDFs covering nearly all the sector
performance variables the model needs. One quarterly PDF contains data for
all 14 variables listed below. You need ~40 PDFs covering 2015Q1–2024Q4.

**Archive URL:** https://nerc.gov.ng/resource-category/nerc-reports/

**What to extract from each PDF:**

| Variable | NERC Table / Section | Units |
|---|---|---|
| `demand_mw` | "Operational Performance" → Peak demand | MW |
| `gas_to_power` | "Gas Supply to Power Sector" | mmscfd |
| `domestic_alloc` | Gas table → Domestic / Total ratio | % |
| `forced_outage` | "GENCO Performance" → Forced Outage Rate | % |
| `eaf` | "GENCO Performance" → Equivalent Availability Factor | % |
| `freq_collapse` | "Grid Stability" → System collapse incidents | count/quarter |
| `tcn_curtailment` | "TCN Performance" → Curtailment | MW |
| `wheeling_cap` | "TCN Performance" → Available wheeling capacity | MW |
| `pipeline_down` | "Gas Supply Constraints" → Pipeline downtime | days/quarter |
| `nbet_pay_ratio` | "Commercial Performance" → NBET payment ratio | % |
| `disco_collect` | "DISCO Performance" → Collection efficiency | % |
| `market_shortfall` | "Market Settlement" → Total shortfall | NGN bn |
| `receivables` | "Market Settlement" → Total receivables | NGN bn |
| `grid_voltage_dev` | "Grid Quality" → Voltage deviation | % |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — NERC QUARTERLY DATA (14 variables)
# Manual extraction from PDF reports
# ═══════════════════════════════════════════════════════════════════════

print('NERC QUARTERLY REPORTS — 14 VARIABLES')
print('='*65)
print()
print('REPORT ARCHIVE: https://nerc.gov.ng/resource-category/nerc-reports/')
print()
print('EXTRACTION WORKFLOW:')
print('  1. Download all quarterly PDFs 2015Q1 → 2024Q4 (~40 reports)')
print('  2. Open each PDF — key tables are:')
print('     • Section 3: "Operational Performance" (generation, demand)')
print('     • Section 4: "Gas Supply" (gas-to-power, pipeline incidents)')
print('     • Section 5: "GENCO Performance" (outage rates, EAF, plant ages)')
print('     • Section 6: "TCN Performance" (curtailment, wheeling, grid quality)')
print('     • Section 7: "DISCO Performance" (collection efficiency)')
print('     • Section 8: "Commercial/Market Settlement" (payment ratios, shortfall)')
print()
print('OPTIONAL PYTHON EXTRACTION (saves time):')
print('  pip install pdfplumber')
print('  import pdfplumber')
print('  with pdfplumber.open("nerc_q1_2024.pdf") as pdf:')
print('      for page in pdf.pages:')
print('          tables = page.extract_tables()')
print('          for t in tables: print(t)')
print()
print('FREQUENCY NOTE:')
print('  NERC reports are quarterly — you get Q1/Q2/Q3/Q4 averages.')
print('  For monthly granularity: take the quarterly average and assign to')
print('  each month within the quarter (constant within quarter).')
print('  TCN publishes DAILY generation reports at tcn.org.ng for demand_mw.')
print()
print('RECENT REPORT DIRECT LINKS:')
print('  Q1 2024: https://nerc.gov.ng/wp-content/uploads/2024/07/2024_Q1-Report_v1YA.pdf')
print('  Q3 2024: https://nerc.gov.ng/media/nerc-third-quarter-2024-report/')

# Create templates for all 14 NERC variables
nerc_vars = {
    'gas_to_power'              : 'Gas supply to power sector (mmscfd) — Section 4',
    'domestic_alloc'            : 'Domestic gas as % of total production — Section 4',
    'pipeline_down'             : 'Gas pipeline downtime (days/quarter) — Section 4',
    'PAF'                       : 'Gas Plant Availability Factor',
    'Transmission_Loss_factor'  : 'Energy sent from GenCo loss via transmission',
    'freq_collapse'             : 'Grid frequency collapse incidents/quarter — Section 6',
    'load_variance'             : 'Difference in load',
    'ATC&C'                     : 'Technical, Commercial and collection loss',
    'Technical_&_commercial_loss': 'part of ATC&C',
    'collection_loss'           : 'part of ATC&C',
    'Total_E_Rec_by_Disco'      : 'total energy sold by nbet & MO to Disco',
    'Total_E_Billed_by_Disco'   : 'total energy billed by Disco to Customers',
    'Total_E_paid_to_Disco'     : 'total remmitance to disco',
    'market_shortfall'          : 'Market settlement shortfall (NGN bn) — Section 8',
    'disco_collect_efficiency'  : 'DISCO collection efficiency % — Section 7',
    'Billing_efficiency'        : 'Efficiency in billing bought energy',
    'nbet_mo_pay_ratio'         : 'NBET payment ratio % — Section 8'
}

# Build multi-column template
tmpl = pd.DataFrame({'Date': dates_idx})
for vname in nerc_vars:
    tmpl[vname] = None

tmpl_path = os.path.join(OUTPUT_DIR, 'nerc_quarterly_TEMPLATE.csv')
tmpl.to_csv(tmpl_path, index=False)
print()
print(f'  ✓ Template saved: nerc_quarterly_TEMPLATE.csv ({len(nerc_vars)} variables)')
print('  Fill this template from NERC PDFs, then rename to nerc_quarterly.csv')

---
## Cell 9 — GENCo Plant Age and Technical Parameters (MANUAL)
**Source:** NERC GENCO licence register + individual GENCO annual reports
**URL:** https://nerc.gov.ng/resource-category/licences/ (Generation licences)

`plant_age_gas` — weighted average age of gas generation fleet (years)
`forced_outage` and `eaf` — covered in NERC quarterly (Cell 8) but plant-level data
comes from individual GENCo performance reports and the NERC MYTO review appendices.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — GENCO PLANT AGE (manual — NERC licence register)
# Model variable: plant_age_gas  |  Units: years (fleet average)
# ═══════════════════════════════════════════════════════════════════════

print('GENCO PLANT AGE — MANUAL DATA')
print('='*65)
print()
print('Source: NERC Generation Licence Register')
print()
print('Key GENCos and approximate commissioning dates:')
gencos = [
    ('Egbin Power Plc',          1985, 1320, 'Steam (CCGT conversion ongoing)'),
    ('Afam VI',                   2009,  630, 'OCGT/CCGT'),
    ('Ughelli Power (Delta)',      1990,  900, 'Gas turbine'),
    ('Geregu Power Plc',          2007,  414, 'OCGT'),
    ('Omotosho Power',            2007,  300, 'OCGT'),
    ('Olorunsogo Power',          2007,  335, 'OCGT'),
    ('Ibom Power',                2009,  190, 'OCGT'),
    ('Kainji Hydro (Mainstream)', 1969, 760,  'Hydro'),
    ('Trans-Amadi Power',         1990,  100, 'Gas turbine'),
]
print(f'  {"GENCo":<30} {"Commiss.":>10} {"Capacity":>10} {"Type"}')
print('  '+'-'*60)
for name, yr, mw, typ in gencos:
    age_2020 = 2020 - yr
    print(f'  {name:<30} {yr:>10} {mw:>9}MW  {typ}')
print()
print('Weighted average fleet age (MW-weighted, at Dec 2024):')
total_mw = sum(mw for _,_,mw,_ in gencos)
avg_age = sum((2024-yr)*mw for _,yr,mw,_ in gencos) / total_mw
print(f'  Approximate: {avg_age:.1f} years (weighted by installed capacity)')
print()
# Build a linear trend as proxy: fleet was ~30yr in 2015, ~40yr in 2024
plant_age_series = pd.Series(index=dates_idx,
    data=np.linspace(30, 40, N))
save_csv(plant_age_series, 'plant_age_gas.csv', 'Gas fleet age (proxy)')
print('  Proxy series saved as plant_age_gas.csv (linear 30→40 years, 2015–2024)')
print('  For exact values: extract from NERC MYTO cost review appendices')


---
## Cell 10 — Pipeline Vandalism and Security Incidents (MANUAL)
**Sources:**
- NERC quarterly reports (Section 4: Gas supply constraints, force majeure)
- Nigerian Upstream Petroleum Regulatory Commission (NUPRC): https://www.nuprc.gov.ng
- Nigeria Extractive Industries Transparency Initiative (NEITI): https://www.neiti.gov.ng

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — PIPELINE VANDALISM (manual + proxy)
# Model variables: pipeline_down, vandalism_idx, fm_dummy
# Units: days/month, count/month, binary
# ═══════════════════════════════════════════════════════════════════════

print('PIPELINE VANDALISM DATA — MANUAL SOURCES')
print('='*65)
print()
print('PRIMARY SOURCE: NERC Quarterly Reports')
print('  Section: "Gas Supply Constraints"')
print('  Variables: pipeline downtime (days), force majeure declarations')
print('  URL: https://nerc.gov.ng/resource-category/nerc-reports/')
print()
print('SECONDARY SOURCE: NNPC Annual Statistical Bulletin')
print('  URL: https://www.nnpcgroup.com/NNPC-Business/Business-Information/Pages/Statistics-Bulletin.aspx')
print('  Table: Gas pipeline incidents by region')
print()
print('TERTIARY: NEITI Reports (annual, verifiable figures)')
print('  URL: https://www.neiti.gov.ng/reports')
print('  Covers: oil theft, pipeline breaches, production losses')
print()
print('Vandalism incident index: no official monthly database exists.')
print('Build proxy from:')
print('  1. NERC quarterly force majeure section (FM events)')
print('  2. NNPC monthly bulletin (pipeline repair reports)')
print('  3. Media aggregation of Niger Delta incidents (for index calibration)')
print()
print('TEMPLATE COLUMNS:')
print('  Date | pipeline_down_days | vandalism_count | fm_declared (0/1)')

# Build proxy from seasonal pattern (dry season = higher vandalism historically)
# Calibrated from NEITI 2015-2020 reports showing seasonal patterns
month_arr = np.array([d.month for d in dates_idx])
pipeline_proxy = pd.Series(index=dates_idx,
    data=np.clip(3 + 4*((month_arr>=11)|(month_arr<=2)).astype(float)
                 + np.random.gamma(1.5, 1, N), 0, 25))
vandalism_proxy = pd.Series(index=dates_idx,
    data=np.clip(np.random.poisson(3, N) + 3*((month_arr>=11)|(month_arr<=2)).astype(float), 0, 20))
fm_proxy = (pipeline_proxy > 15).astype(float)

tmpl_vandalism = pd.DataFrame({
    'Date': dates_idx,
    'pipeline_down': None,
    'vandalism_idx': None,
    'fm_dummy': None,
})
tmpl_vandalism.to_csv(os.path.join(OUTPUT_DIR, 'vandalism_TEMPLATE.csv'), index=False)
save_csv(pipeline_proxy, 'pipeline_down.csv', 'Pipeline downtime proxy (replace with NERC data)')
save_csv(vandalism_proxy, 'vandalism_idx.csv', 'Vandalism index proxy (replace with NERC/NNPC data)')
save_csv(fm_proxy, 'fm_dummy.csv', 'Force majeure binary (derived from pipeline_down>15 days)')
print()
print('  Proxy series saved. Replace with extracted NERC/NNPC data.')

---
## Cell 11 — NUPRC Gas Production and NLNG Utilisation (MANUAL)
**Source:** NUPRC Monthly Gas Production Status Report
**URL:** https://www.nuprc.gov.ng (search "gas production status report")
**Also:** NNPC Annual Statistical Bulletin and GIIGNL Annual Report

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 11 — NUPRC GAS DATA
# Model variables: gas_to_power, domestic_alloc, nlng_util
# (gas_to_power and domestic_alloc also in NERC Cell 8 — prefer NUPRC)
# ═══════════════════════════════════════════════════════════════════════

print('NUPRC GAS PRODUCTION DATA — MANUAL EXTRACTION')
print('='*65)
print()
print('PRIMARY SOURCE: NUPRC Monthly Gas Production Status Report')
print('  URL: https://www.nuprc.gov.ng')
print('  Search: "gas production status report" or "monthly production"')
print()
print('DIRECT 2025 PDF:')
print('  https://www.nuprc.gov.ng/wp-content/uploads/2025/08/JAN-TO-DEC-2025-PRODUCTION.pdf')
print()
print('WHAT TO EXTRACT FROM EACH MONTHLY PDF:')
print('  Variable         | NUPRC Column Name            | Unit')
print('  ─────────────────────────────────────────────────────────')
print('  gas_to_power     | Gas for Power (Domestic)     | mmscfd')
print('  domestic_alloc   | Domestic / Total Produced    | fraction 0–1')
print('  nlng_util        | LNG Export / 2,920 mmscfd    | fraction 0–1')
print()
print('NLNG nameplate capacity calculation:')
print('  22 mmtpa LNG × 1.36 Btu/scf conversion ÷ 365 days ≈ 2,920 mmscfd max throughput')
print('  Utilisation = Actual LNG export (mmscfd) / 2,920')
print()
print('SUPPLEMENTARY SOURCES:')
print('  GIIGNL Annual Report: https://giignl.org/lng-industry/')
print('  → Nigeria LNG export volumes by year (used for annual validation)')
print('  NNPC Bulletin: https://www.nnpcgroup.com → Business Info → Statistics')

tmpl_nuprc = pd.DataFrame({
    'Date': dates_idx,
    'gas_to_power_mmscfd': None,
    'domestic_alloc_fraction': None,
    'nlng_util_fraction': None,
})
tmpl_nuprc.to_csv(os.path.join(OUTPUT_DIR, 'nuprc_gas_TEMPLATE.csv'), index=False)
print()
print('  Template saved: nuprc_gas_TEMPLATE.csv')

---
## Cell 12 — Derived and Binary Variables (AUTO — computed from other series)
These variables require no external download. They are computed deterministically
from other series or from project-specific inputs.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 12 — DERIVED VARIABLES (computed — no download needed)
# Variables: harmattan, grid_demand_norm, dgso_dummy,
#            fm_dummy, ln_usdngn_centred, solar_plant_age
# ═══════════════════════════════════════════════════════════════════════

print('Computing derived variables...')

# ── Harmattan dust season binary (Nov–Mar, northern Nigeria) ──────────────
month_idx = np.array([d.month for d in dates_idx])
harmattan = pd.Series(index=dates_idx,
    data=((month_idx>=11)|(month_idx<=3)).astype(float))
save_csv(harmattan, 'harmattan.csv', 'Harmattan dummy (Nov=1, Mar=1, else 0)')

# ── DGSO enforcement binary ───────────────────────────────────────────────
# Domestic Gas Supply Obligation formally enforced from January 2016
# Source: NNPC DGSO policy | Gas flaring regulations 2018 also strengthened
dgso = pd.Series(index=dates_idx,
    data=(dates_idx >= '2016-01-01').astype(float))
save_csv(dgso, 'dgso_dummy.csv', 'DGSO enforcement binary (1 from Jan 2016)')

# ── Solar plant age (years since commissioning) ───────────────────────────
# Set plant commissioning date to match your project
# Default: 2015-01-01 (Nigerian utility-scale solar build-out began ~2015)
# Nigeria solar timeline: first utility projects ~2015, major expansion 2018+
SOLAR_COMMISSION_DATE = pd.Timestamp('2015-01-01')
solar_age = pd.Series(index=dates_idx,
    data=[(d - SOLAR_COMMISSION_DATE).days / 365.25 for d in dates_idx])
solar_age = solar_age.clip(lower=0)
save_csv(solar_age, 'solar_plant_age.csv', 'Solar plant age (years since 2015-01-01)')

# ── ln_usdngn_centred ─────────────────────────────────────────────────────
# Computed once usdngn.csv exists
if os.path.exists(os.path.join(OUTPUT_DIR, 'usdngn.csv')):
    usdngn_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'usdngn.csv'), index_col=0, parse_dates=True).squeeze()
    usdngn_df = usdngn_df.reindex(dates_idx, method='ffill')
    ln_usdngn_c = np.log(usdngn_df.clip(lower=1e-3)) - np.log(300)
    # Centred at NGN/USD=300 (2015 base). At NGN=197: -0.42. At NGN=1500: +1.61.
    # This prevents ln(1500)=7.31 from dominating the solar CF regression.
    save_csv(ln_usdngn_c, 'ln_usdngn_centred.csv', 'Centred log FX (base=ln(300))')
    print(f'  ln_usdngn_centred range: {ln_usdngn_c.min():.2f} – {ln_usdngn_c.max():.2f}')
    print(f'  At NGN=300: 0.000 | At NGN=1500: {np.log(1500)-np.log(300):.3f}')
else:
    print('  usdngn.csv not found — run Cell 2 first, then re-run this cell')

# ── Force majeure binary (derived from pipeline downtime) ─────────────────
if os.path.exists(os.path.join(OUTPUT_DIR, 'pipeline_down.csv')):
    pipe = pd.read_csv(os.path.join(OUTPUT_DIR, 'pipeline_down.csv'), index_col=0, parse_dates=True).squeeze()
    fm_derived = (pipe.reindex(dates_idx, method='ffill') > 15).astype(float)
    save_csv(fm_derived, 'fm_dummy.csv', 'Force majeure binary (pipeline_down > 15 days)')
else:
    print('  pipeline_down.csv not found — fm_dummy will be computed when available')

# ── JKM netback (computed from jkm_prices.csv) ────────────────────────────
if os.path.exists(os.path.join(OUTPUT_DIR, 'jkm_prices.csv')):
    jkm_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'jkm_prices.csv'), index_col=0, parse_dates=True).squeeze()
    jkm_nb = (jkm_df.reindex(dates_idx, method='ffill') - 2.5).clip(lower=0.5)
    save_csv(jkm_nb, 'jkm_netback.csv', 'JKM netback (JKM - $2.50/MMBtu shipping)')
else:
    print('  jkm_prices.csv not found — jkm_netback will be computed when available')

print()
print('Derived variables: harmattan, dgso_dummy, solar_plant_age computed.')
print('ln_usdngn_centred, fm_dummy, jkm_netback computed if dependencies present.')

---
## Cell 13 — Master Merge and Quality Report
Merges all available series into `master_panel.csv`.
Shows completeness per variable so you know exactly what still needs manual entry.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 14 — MASTER MERGE AND QUALITY REPORT
# ═══════════════════════════════════════════════════════════════════════

# All files and their model variable names
file_map = {
    'brent_prices.csv'       : ('brent_usd_bbl', 'brent'),
    'usdngn.csv'             : ('usdngn', 'usdngn'),
    'ttf_jkm_spread.csv'     : ('0', 'ttf_jkm'),
    'jkm_netback.csv'        : ('0', 'jkm_netback'),
    'ghi.csv'                : ('ALLSKY_SFC_SW_DWN', 'ghi'),
    'dni.csv'                : ('ALLSKY_SFC_SW_DNI', 'dni'),
    'temperature.csv'        : ('T2M', 'temp_c'),
    'precipitation.csv'      : ('PRECTOTCORR', 'precip_mm'),
    'aerosol_od.csv'         : ('0', 'aerosol_od'),
    'fx_reserves.csv'        : ('fx_reserves_usd_bn', 'fx_reserves'),
    'inflation.csv'          : ('0', 'inflation'),
    'ng_gas_price.csv'       : ('0', 'gas_price'),
    'harmattan.csv'          : ('0', 'harmattan'),
    'dgso_dummy.csv'         : ('0', 'dgso_dummy'),
    'solar_plant_age.csv'    : ('0', 'solar_plant_age'),
    'ln_usdngn_centred.csv'  : ('0', 'ln_usdngn_centred'),
    'fm_dummy.csv'           : ('0', 'fm_dummy'),
    'pipeline_down.csv'      : ('0', 'pipeline_down'),
    'vandalism_idx.csv'      : ('0', 'vandalism_idx'),
    'plant_age_gas.csv'      : ('0', 'plant_age_gas'),
    'nlng_utilization.csv'   : ('utilisation_pct', 'nlng_util'),
}
# NERC quarterly file covers multiple variables
nerc_vars = ['gas_to_power','domestic_alloc','pipeline_down','PAF','Transmission_Loss_factor',
             'freq_collapse','load_variance','ATC&C','Tech_&_commercial_loss','collection_loss',
             'Total_E_Rec_by_disco','Total_E_Billed_by_Disco','Total_E_paid_to_Disco','market_shortfall',
             'disco_collect_efficiency','Billing_efficiency','nbet_mo_pay_ratio']

master = pd.DataFrame(index=dates_idx)

print('MASTER PANEL STATUS REPORT')
print('='*72)
print(f'{"Variable":<22} {"File":<30} {"Status":>8} {"Complete%":>10}')
print('-'*72)

for fname, (col_hint, model_var) in file_map.items():
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        try:
            df_raw = pd.read_csv(fpath, index_col=0, parse_dates=True)
            # Pick the right column
            if col_hint in df_raw.columns: s = df_raw[col_hint]
            elif len(df_raw.columns) == 1: s = df_raw.iloc[:,0]
            else: s = df_raw.iloc[:,0]
            s_aligned = s.reindex(dates_idx, method='ffill')
            master[model_var] = s_aligned
            pct = s_aligned.notna().mean()*100
            flag = '✓ REAL' if not fname.endswith('_TEMPLATE.csv') else '⚠ PROXY'
            print(f'{model_var:<22} {fname:<30} {flag:>8} {pct:>9.0f}%')
        except Exception as ex:
            print(f'{model_var:<22} {fname:<30} {"✗ ERROR":>8}  {str(ex)[:20]}')
    else:
        print(f'{model_var:<22} {fname:<30} {"MISSING":>8}   0%')

# NERC quarterly file
nerc_path = os.path.join(OUTPUT_DIR, 'nerc_quarterly_TEMPLATE.csv')
for vname in nerc_vars:
    if vname not in master.columns:
        if os.path.exists(nerc_path):
            try:
                nerc_df = pd.read_csv(nerc_path, index_col=0, parse_dates=True)
                nerc_df = nerc_df[~nerc_df.index.isna()]
                if vname in nerc_df.columns:
                    master[vname] = nerc_df[vname].reindex(dates_idx, method='ffill')
                    pct = master[vname].notna().mean()*100
                    print(f'{vname:<22} {"nerc_quarterly.csv":<30} {"✓ REAL":>8} {pct:>9.0f}%')
                else:
                    print(f'{vname:<22} {"nerc_quarterly.csv":<30} {"MISSING":>8}   0%')
            except Exception as e:
                print(
                    f'{vname:<22} {"nerc_quarterly.csv":<30} '
                    f'{"✗ ERROR":>8} {repr(e)}'
                )
        else:
            print(f'{vname:<22} {"nerc_quarterly.csv":<30} {"MISSING":>8}   0%')

print('-'*72)
total_vars = len(master.columns)
complete_vars = (master.notna().mean() > 0.8).sum()
print(f'Coverage: {complete_vars}/{total_vars} variables ≥80% complete')

master.to_csv(os.path.join(OUTPUT_DIR, 'master_panel.csv'))
print(f'\n  ✓ Saved: master_panel.csv  ({master.shape[0]} rows × {master.shape[1]} columns)')

In [ ]:
candidate_vars = [
    'gas_to_power',
    'domestic_alloc',
    'PAF',
    'market_shortfall',
    'nbet_mo_pay_ratio',
    'Billing_efficiency'
]

print(
    master[candidate_vars]
    .dropna()
    .shape
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 14 — GRID_DEMAND_NORM (computed from demand_mw)
# Model variable: grid_demand_norm  |  (demand_mw - mean) / std
# Must be computed AFTER demand_mw is available
# ═══════════════════════════════════════════════════════════════════════

if 'demand_mw' in master.columns and master['demand_mw'].notna().sum() > 12:
    dm = master['demand_mw'].ffill()
    gdn = (dm - dm.mean()) / dm.std()
    master['grid_demand_norm'] = gdn
    save_csv(gdn, 'grid_demand_norm.csv', 'Grid demand (normalised)')
    print(f'grid_demand_norm: mean={gdn.mean():.4f} std={gdn.std():.4f}')
    # Re-save master with this column
    master.to_csv(os.path.join(OUTPUT_DIR, 'master_panel.csv'))
else:
    print('demand_mw not yet available — fill nerc_quarterly.csv first, then re-run')

In [ ]:
panel = pd.read_csv("master_panel.csv", index_col=0, parse_dates=True)
print(sorted(panel.columns.tolist()))

---
## Cell 16 — Pipeline Completion Status Summary

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 16 — PIPELINE COMPLETION SUMMARY
# ═══════════════════════════════════════════════════════════════════════

print('PIPELINE COMPLETION SUMMARY')
print('='*65)

auto_vars = ['brent','usdngn','ttf_jkm','jkm_netback','ghi','dni','temp_c',
             'precip_mm','aerosol_od','fx_reserves','inflation',
             'harmattan','dgso_dummy','solar_plant_age','ln_usdngn_centred','fm_dummy']
manual_vars = ['gas_price','demand_mw','gas_to_power','domestic_alloc',
               'nlng_util','forced_outage','eaf','freq_collapse','tcn_curtailment',
               'wheeling_cap','pipeline_down','vandalism_idx','plant_age_gas',
               'nbet_pay_ratio','disco_collect','market_shortfall','receivables',
               'grid_voltage_dev','grid_demand_norm']

print('\nAUTO (run pipeline and these download):')
for v in auto_vars:
    done = v in master.columns and master[v].notna().mean() > 0.5
    print(f'  {"✓" if done else "✗"} {v}')

print('\nMANUAL (requires PDF extraction / manual download):')
for v in manual_vars:
    done = v in master.columns and master[v].notna().mean() > 0.5
    src_map = {
        'gas_price': 'NMDPRA + NERC (Cell 7)',
        'demand_mw': 'NERC quarterly (Cell 8)', 'gas_to_power':'NUPRC (Cell 11)',
        'domestic_alloc':'NUPRC (Cell 11)','nlng_util':'NUPRC (Cell 11)',
        'forced_outage':'NERC quarterly (Cell 8)','eaf':'NERC quarterly (Cell 8)',
        'freq_collapse':'NERC quarterly (Cell 8)','tcn_curtailment':'NERC quarterly (Cell 8)',
        'wheeling_cap':'NERC quarterly (Cell 8)','pipeline_down':'NERC/NNPC (Cell 10)',
        'vandalism_idx':'NERC/NNPC (Cell 10)','plant_age_gas':'NERC GENCO (Cell 9)',
        'nbet_pay_ratio':'NERC quarterly (Cell 8)','disco_collect':'NERC quarterly (Cell 8)',
        'market_shortfall':'NERC quarterly (Cell 8)','receivables':'NERC quarterly (Cell 8)',
        'grid_voltage_dev':'TCN/NERC (Cell 13)','grid_demand_norm':'Computed from demand_mw (Cell 15)',
    }
    src = src_map.get(v,'')
    print(f'  {"✓" if done else "○"} {v:<22} ← {src}')

n_done = sum(1 for v in auto_vars+manual_vars
             if v in master.columns and master[v].notna().mean()>0.5)
print(f'\nTotal: {n_done}/{len(auto_vars)+len(manual_vars)} variables complete')
print('\nAll NERC quarterly variables (14 vars) come from the same PDF series.')
print('One afternoon extracting 40 quarterly PDFs completes all manual variables.')

---
## Cell 17 — Integration Guide: How to Use This Pipeline in the Model

**This is the critical cell. Read it before opening the model notebook.**


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 17 — INTEGRATION GUIDE
# How to replace synthetic data in integrated_gas_solar_model.ipynb
# with real data from this pipeline
# ═══════════════════════════════════════════════════════════════════════

guide = """
╔══════════════════════════════════════════════════════════════════════╗
║          INTEGRATION GUIDE — PIPELINE → MODEL                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  STEP 1: Run all cells in this pipeline notebook (data_pipeline.ipynb)
║          Cells 1–6 auto-download in minutes.                        ║
║          Cells 7–13 produce templates — fill from NERC PDFs.        ║
║          Cell 14 produces master_panel.csv                          ║
║                                                                      ║
║  STEP 2: Open integrated_gas_solar_model.ipynb                      ║
║                                                                      ║
║  STEP 3: REPLACE Cell 1 (Data Assembly) with Cell 18 below.         ║
║          Cell 18 loads master_panel.csv and constructs the          ║
║          exact same df DataFrame that Cell 1 builds synthetically.  ║
║          All downstream cells (2–17) run unchanged.                 ║
║                                                                      ║
║  STEP 4: Re-run from Cell 1 downward.                               ║
║          ADF tests, cointegration, OLS, SUR, MC all use real data.  ║
║                                                                      ║
║  PARTIAL DATA OPTION:                                               ║
║  Cell 18 has a hybrid mode: if a variable is in master_panel.csv,   ║
║  it uses the real series. If missing, it falls back to the          ║
║  calibrated synthetic series from Cell 1. This means you can go    ║
║  live with just the AUTO variables (brent, FX, irradiance) while   ║
║  still building the NERC manual series over time.                   ║
║                                                                      ║
║  VARIABLE ALIGNMENT RULES:                                          ║
║  • All series must be monthly (MS frequency), 2015-01 to 2024-12   ║
║  • master_panel.csv column names must match model variable names    ║
║    exactly (see Cell 14 for the column name mapping)               ║
║  • ln_usdngn_centred is computed from usdngn in Cell 12 —          ║
║    do not put it in master_panel.csv manually                       ║
║  • grid_demand_norm is computed from demand_mw in Cell 15           ║
║  • harmattan, dgso_dummy, fm_dummy, solar_plant_age are computed    ║
║    deterministically — do not override them                         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(guide)

In [1]:
# --------------------------------------------------
# RESTRICT ANALYSIS PERIOD
# --------------------------------------------------
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer

panel = pd.read_csv(
    "master_panel.csv",
    index_col=0,
    parse_dates=True
)

panel = panel.loc[
    panel.index >= "2017-01-01"
].copy()

print(
    f"Analysis period: "
    f"{panel.index.min().date()} "
    f"to "
    f"{panel.index.max().date()}"
)

print(
    f"Observations: {len(panel)}"
)

raw_panel = panel.copy()

Analysis period: 2017-01-01 to 2026-05-01
Observations: 113


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 15B (PART 1)
# PHYSICS-INFORMED RECONSTRUCTION ENGINE
#
# PART 1:
# LOAD → CLEAN → INTERPOLATE → IDENTITY RECOVERY
#
# NO RIDGE RECONSTRUCTION YET
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer

# ──────────────────────────────────────────────────────────────
# LOAD PANEL
# ──────────────────────────────────────────────────────────────

#panel = pd.read_csv(
 #   "master_panel.csv",
  #  index_col=0,
   # parse_dates=True
#)

print(
    f"Loaded panel: "
    f"{panel.shape[0]} rows × {panel.shape[1]} columns"
)

# ──────────────────────────────────────────────────────────────
# HARMONIZE VARIABLE NAMES
# ──────────────────────────────────────────────────────────────

panel = panel.rename(columns={
    "PAF": "eaf",
    "nbet_mo_pay_ratio": "nbet_pay_ratio",
    "disco_collect_efficiency": "disco_collect"
})

# ──────────────────────────────────────────────────────────────
# REPORT STRUCTURE
# ──────────────────────────────────────────────────────────────

report_rows = []

for col in panel.columns:

    report_rows.append({
        "variable": col,
        "original_completeness_pct":
            round(panel[col].notna().mean() * 100, 1)
    })

# ──────────────────────────────────────────────────────────────
# FORCE NUMERIC TYPES
# ──────────────────────────────────────────────────────────────

for col in panel.columns:

    panel[col] = (
        panel[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    panel[col] = pd.to_numeric(
        panel[col],
        errors="coerce"
    )

print("\nNON-NUMERIC CHECK")

for col in panel.columns:

    if not pd.api.types.is_numeric_dtype(
        panel[col]
    ):
        print(
            f"{col} : {panel[col].dtype}"
        )

# ──────────────────────────────────────────────────────────────
# VARIABLE GROUPS
# ──────────────────────────────────────────────────────────────

INTERP_VARS = [

    "eaf",
    "Billing_efficiency",
    "disco_collect",
    "Tech_&_commercial_loss",
    "collection_loss",
    "Total_E_Billed_by_Disco",
    "Total_E_paid_to_Disco",
    "Total_E_Rec_by_disco",
    "solar_cf"
]

MACRO_SERIES = [

    "usdngn",
    "brent",
    "inflation",
    "fx_reserves",
    "ttf_jkm",
    "jkm_netback"
]

# ──────────────────────────────────────────────────────────────
# STEP 1
# INTERPOLATION
# ──────────────────────────────────────────────────────────────

print("\nINTERPOLATING VARIABLES")

for col in INTERP_VARS:

    if col not in panel.columns:
        continue

    original = panel[col].copy()

    filled = (
        panel[col]
        .interpolate(method="time")
        .ffill()
        .bfill()
    )

    panel.loc[
        original.isna(),
        col
    ] = filled.loc[
        original.isna()
    ]

# ──────────────────────────────────────────────────────────────
# STEP 1B
# MACRO SERIES
# ──────────────────────────────────────────────────────────────

for col in MACRO_SERIES:

    if col not in panel.columns:
        continue

    panel[col] = (
        panel[col]
        .interpolate(method="linear")
        .ffill()
        .bfill()
    )

# ──────────────────────────────────────────────────────────────
# STEP 2
# IDENTITY RECOVERY
# PRIORITY:
# REAL DATA > IDENTITY > RECONSTRUCTION
# ──────────────────────────────────────────────────────────────

print("\nRECOVERING ACCOUNTING IDENTITIES")

# --------------------------------------------------
# BILLING EFFICIENCY
# --------------------------------------------------

if all(
    c in panel.columns
    for c in [
        "Total_E_Billed_by_Disco",
        "Total_E_Rec_by_disco"
    ]
):

    mask = (
        panel["Billing_efficiency"].isna()
        &
        panel["Total_E_Billed_by_Disco"].notna()
        &
        panel["Total_E_Rec_by_disco"].notna()
        &
        (panel["Total_E_Rec_by_disco"] > 0)
    )

    panel.loc[
        mask,
        "Billing_efficiency"
    ] = (
        100
        * panel.loc[
            mask,
            "Total_E_Billed_by_Disco"
        ]
        /
        panel.loc[
            mask,
            "Total_E_Rec_by_disco"
        ]
    )

# --------------------------------------------------
# COLLECTION EFFICIENCY
# --------------------------------------------------

if all(
    c in panel.columns
    for c in [
        "Total_E_paid_to_Disco",
        "Total_E_Billed_by_Disco"
    ]
):

    mask = (
        panel["disco_collect"].isna()
        &
        panel["Total_E_paid_to_Disco"].notna()
        &
        panel["Total_E_Billed_by_Disco"].notna()
        &
        (panel["Total_E_Billed_by_Disco"] > 0)
    )

    panel.loc[
        mask,
        "disco_collect"
    ] = (
        100
        * panel.loc[
            mask,
            "Total_E_paid_to_Disco"
        ]
        /
        panel.loc[
            mask,
            "Total_E_Billed_by_Disco"
        ]
    )

# --------------------------------------------------
# COLLECTION LOSS
# --------------------------------------------------

if all(
    c in panel.columns
    for c in [
        "collection_loss",
        "disco_collect"
    ]
):

    mask = (
        panel["collection_loss"].isna()
        &
        panel["disco_collect"].notna()
    )

    panel.loc[
        mask,
        "collection_loss"
    ] = (
        100
        - panel.loc[
            mask,
            "disco_collect"
        ]
    )

# --------------------------------------------------
# ATC&C
# Formula:
# 100 * (1 - (BE * CE / 10000))
# --------------------------------------------------

if all(
    c in panel.columns
    for c in [
        "ATC&C",
        "Billing_efficiency",
        "disco_collect"
    ]
):

    mask = (
        panel["ATC&C"].isna()
        &
        panel["Billing_efficiency"].notna()
        &
        panel["disco_collect"].notna()
    )

    panel.loc[
        mask,
        "ATC&C"
    ] = (
        100
        * (
            1
            -
            (
                panel.loc[
                    mask,
                    "Billing_efficiency"
                ]
                *
                panel.loc[
                    mask,
                    "disco_collect"
                ]
                / 10000
            )
        )
    )

# --------------------------------------------------
# TECHNICAL + COMMERCIAL LOSS
# --------------------------------------------------

if all(
    c in panel.columns
    for c in [
        "Tech_&_commercial_loss",
        "ATC&C",
        "collection_loss"
    ]
):

    mask = (
        panel["Tech_&_commercial_loss"].isna()
        &
        panel["ATC&C"].notna()
        &
        panel["collection_loss"].notna()
    )

    panel.loc[
        mask,
        "Tech_&_commercial_loss"
    ] = (
        panel.loc[
            mask,
            "ATC&C"
        ]
        -
        panel.loc[
            mask,
            "collection_loss"
        ]
    )

# ──────────────────────────────────────────────────────────────
# COMPLETENESS CHECK AFTER IDENTITIES
# ──────────────────────────────────────────────────────────────

print("\nPOST-IDENTITY COMPLETENESS")

for v in [
    "Billing_efficiency",
    "disco_collect",
    "collection_loss",
    "ATC&C",
    "Tech_&_commercial_loss"
]:

    if v in panel.columns:

        print(
            f"{v:<30}"
            f"{panel[v].notna().mean()*100:6.1f}%"
        )

print("\nPART 1 COMPLETE")
print("Ready for Part 2 (Constrained Reconstruction Engine)")

Loaded panel: 113 rows × 38 columns

NON-NUMERIC CHECK

INTERPOLATING VARIABLES

RECOVERING ACCOUNTING IDENTITIES

POST-IDENTITY COMPLETENESS
Billing_efficiency             100.0%
disco_collect                  100.0%
collection_loss                100.0%
ATC&C                          100.0%
Tech_&_commercial_loss         100.0%

PART 1 COMPLETE
Ready for Part 2 (Constrained Reconstruction Engine)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 15B (PART 2)
# CONSTRAINED RECONSTRUCTION ENGINE
# ═══════════════════════════════════════════════════════════════════════

from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer

# ──────────────────────────────────────────────────────────────
# RECONSTRUCTION RULES
# ──────────────────────────────────────────────────────────────

RECON_RULES = {

    "market_shortfall": {
        "predictors": [
            "Total_E_Rec_by_disco",
            "Total_E_paid_to_Disco",
            "disco_collect",
            "nbet_pay_ratio"
        ],

        "min": 0,
        "max": None,
        "integer": False
    },

    "Transmission_Loss_factor": {

        "predictors": [
            "vandalism_idx",
            "freq_collapse",
            "eaf",
            "ATC&C",
        ],

        "min": 0,
        "max": 10,
        "integer": False
    },

    "nbet_pay_ratio": {

      "predictors": [
            "disco_collect",
            "Billing_efficiency",
            "ATC&C",
            "Total_E_Rec_by_disco",
            "Total_E_paid_to_Disco"
        ],

        "min": 20,
        "max": 45,
        "integer": False
    },

    "gas_to_power": {

        "predictors": [
            "gas_price",
            "usdngn",
            "pipeline_down",
            "vandalism_idx",
            "market_shortfall",
            "nbet_pay_ratio",
            "freq_collapse",
            "Transmission_Loss_factor"
        ],

        "min": 0,
        "max": None,
        "integer": False
    },

    "domestic_alloc": {

        "predictors": [
            "gas_to_power",
            "gas_cf",
            "jkm_netback",
            "ttf_jkm",
            "Total_E_Rec_by_disco"
        ],

        "min": 0,
        "max": None,
        "integer": False
    },

    "load_variance": {

        "predictors": [
            "ATC&C",
            "disco_collect",
            "market_shortfall",
            "freq_collapse",
            "Transmission_Loss_factor",
            "Billing_efficiency"
        ],

        "min": 0,
        "max": None,
        "integer": False
    },

    "gas_cf": {

        "predictors": [
            "eaf",
            "gas_to_power",
            "domestic_alloc",
            "pipeline_down",
            "vandalism_idx",
            "freq_collapse",
            "Total_E_Rec_by_disco"
        ],

        "min": 0,
        "max": 100,
        "integer": False
    },

    "Gas_constraint": {

        "predictors": [
            "pipeline_down",
            "vandalism_idx",
            "gas_to_power",
            "domestic_alloc",
            "gas_cf"
        ],

        "min": 0,
        "max": None,
        "integer": False
    },

    "Distribution_constraint": {

        "predictors": [
            "ATC&C",
            "Billing_efficiency",
            "disco_collect",
            "load_variance",
            "Total_E_Billed_by_Disco"
        ],

        "min": 0,
        "max": None,
        "integer": False
    },

    "Transmission_constraint": {

        "predictors": [
            "Transmission_Loss_factor",
            "freq_collapse",
            "Total_E_Rec_by_disco",
            "gas_cf"
        ],

        "min": 0,
        "max": None,
        "integer": False
    }
}

# ──────────────────────────────────────────────────────────────
# GENERIC RECONSTRUCTION FUNCTION
# ──────────────────────────────────────────────────────────────

def constrained_reconstruct(
    df,
    target,
    rule
):

    if target not in df.columns:

        print(
            f"SKIPPED: {target} not found"
        )

        return

    observed_mask = df[target].notna()

    n_obs = observed_mask.sum()

    if n_obs < 4:

        print(
            f"SKIPPED: {target} only "
            f"{n_obs} observations"
        )

        return

    predictors = [

        p for p in rule["predictors"]

        if p in df.columns
    ]

    if len(predictors) == 0:

        print(
            f"SKIPPED: {target} "
            f"no predictors"
        )

        return

    X = df[predictors].copy()

    imp = SimpleImputer(
        strategy="median"
    )

    X_imp = pd.DataFrame(

        imp.fit_transform(X),

        index=X.index,

        columns=X.columns
    )

    X_train = X_imp.loc[
        observed_mask
    ]

    y_train = df.loc[
        observed_mask,
        target
    ]

    model = Ridge(
        alpha=1.0
    )

    model.fit(
        X_train,
        y_train
    )

    predicted = model.predict(
        X_imp
    )

    # --------------------------------------------------
    # HARD BOUNDS
    # --------------------------------------------------

    lower = rule["min"]
    upper = rule["max"]

    predicted = np.clip(
        predicted,
        lower,
        upper
    )

    # --------------------------------------------------
    # INTEGER VARIABLES
    # --------------------------------------------------

    if rule["integer"]:

        predicted = np.round(
            predicted
        )

    # --------------------------------------------------
    # FILL ONLY MISSING
    # --------------------------------------------------

    missing_mask = (
        ~observed_mask
    )

    df.loc[
        missing_mask,
        target
    ] = predicted[
        missing_mask
    ]

    print(
        f"{target:<30}"
        f"{n_obs:>5} obs → "
        f"{df[target].notna().sum():>5} complete"
    )

# ──────────────────────────────────────────────────────────────
# EXECUTE RULES
# ──────────────────────────────────────────────────────────────

print("\n")
print("=" * 72)
print("CONSTRAINED RECONSTRUCTION")
print("=" * 72)

for target, rule in RECON_RULES.items():

    constrained_reconstruct(
        panel,
        target,
        rule
    )

# ──────────────────────────────────────────────────────────────
# POST-RECONSTRUCTION CHECKS
# ──────────────────────────────────────────────────────────────

print("\n")
print("=" * 72)
print("RECONSTRUCTION SUMMARY")
print("=" * 72)

for target in RECON_RULES:

    if target not in panel.columns:
        continue

    print("\n" + target)

    print(
        panel[target]
        .describe()
    )

    if (panel[target] < 0).any():

        print(
            "WARNING: "
            "Negative values detected"
        )

# ──────────────────────────────────────────────────────────────
# INTEGER ENFORCEMENT
# ──────────────────────────────────────────────────────────────

INTEGER_COLUMNS = [

    "freq_collapse",
    "pipeline_down"
]

for col in INTEGER_COLUMNS:

    if col not in panel.columns:
        continue

    panel[col] = (

        panel[col]

        .ffill()

        .bfill()

        .fillna(0)

        .round()

        .clip(lower=0)

        .astype(int)
    )

print("\nPART 2 COMPLETE")
print(
    "Ready for Part 3 "
    "(Economic Audits + Reporting + Export)"
)



CONSTRAINED RECONSTRUCTION
market_shortfall                 83 obs →   113 complete
Transmission_Loss_factor        107 obs →   113 complete
nbet_pay_ratio                  107 obs →   113 complete
gas_to_power                      7 obs →   113 complete
domestic_alloc                    7 obs →   113 complete
load_variance                    12 obs →   113 complete
gas_cf                           90 obs →   113 complete
Gas_constraint                   15 obs →   113 complete
Distribution_constraint          15 obs →   113 complete
Transmission_constraint          15 obs →   113 complete


RECONSTRUCTION SUMMARY

market_shortfall
count    1.130000e+02
mean     5.892570e+10
std      4.932137e+10
min      9.973333e+09
25%      2.862500e+10
50%      4.235621e+10
75%      6.830841e+10
max      1.910000e+11
Name: market_shortfall, dtype: float64

Transmission_Loss_factor
count    113.000000
mean       8.126872
std        0.655060
min        7.190000
25%        7.560000
50%        8.1500

c:\Users\pc\miniconda3\envs\FE\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.5806803221346846e-23.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\pc\miniconda3\envs\FE\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.908070883458872e-24.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\pc\miniconda3\envs\FE\Lib\site-packages\sklearn\linear_model\_ridge.py:267: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
c:\Users\pc\miniconda3\envs\FE\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.79188051155875e-22.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\pc\miniconda3\envs\FE\Lib\site-packages\sklearn\linear_model\_rid

count    113.000000
mean      62.557101
std       24.820786
min       18.250000
25%       41.000000
50%       71.350000
75%       82.370000
max       96.930000
Name: nbet_pay_ratio, dtype: float64

gas_to_power
count      113.000000
mean     27125.897155
std        669.513953
min      23782.360000
25%      27045.337357
50%      27329.381725
75%      27494.997147
max      27722.197480
Name: gas_to_power, dtype: float64

domestic_alloc
count      113.000000
mean     29833.379191
std      21606.137913
min          0.000000
25%      17244.589021
50%      27693.774867
75%      38524.251507
max      84421.043315
Name: domestic_alloc, dtype: float64

load_variance
count    113.000000
mean       0.648758
std        0.994652
min       -0.110000
25%        0.000000
50%        0.040000
75%        0.978271
max        3.511636
Name: load_variance, dtype: float64

gas_cf
count    113.000000
mean      28.371268
std        2.655613
min       22.810000
25%       26.674825
50%       28.400000
75%       

In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 15B (PART 3)
# ECONOMIC AUDITS + REPORTING + EXPORT
# ═══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────
# ECONOMIC AUDITS
# ──────────────────────────────────────────────────────────────

print("\n")
print("=" * 72)
print("ECONOMIC AUDITS")
print("=" * 72)

audit_results = []

def audit_correlation(
    var1,
    var2,
    expected_sign
):

    if (
        var1 not in panel.columns
        or
        var2 not in panel.columns
    ):
        return

    tmp = panel[[var1, var2]].dropna()

    if len(tmp) < 5:
        return

    corr = tmp[var1].corr(
        tmp[var2]
    )

    if expected_sign == "positive":

        status = (
            "PASS"
            if corr > 0
            else "FAIL"
        )

    else:

        status = (
            "PASS"
            if corr < 0
            else "FAIL"
        )

    audit_results.append({

        "Variable_1": var1,
        "Variable_2": var2,
        "Expected": expected_sign,
        "Correlation": round(corr, 4),
        "Status": status
    })

# --------------------------------------------------
# RELATIONSHIP AUDITS
# --------------------------------------------------

audit_correlation(
    "vandalism_idx",
    "pipeline_down",
    "positive"
)

audit_correlation(
    "gas_to_power",
    "gas_cf",
    "positive"
)

audit_correlation(
    "Billing_efficiency",
    "ATC&C",
    "negative"
)

audit_correlation(
    "disco_collect",
    "collection_loss",
    "negative"
)

audit_correlation(
    "Total_E_Rec_by_disco",
    "Transmission_Loss_factor",
    "negative"
)

audit_df = pd.DataFrame(
    audit_results
)

print("\nAUDIT RESULTS")

if len(audit_df):

    print(audit_df)

else:

    print("No audits executed")

# ──────────────────────────────────────────────────────────────
# PHYSICAL CONSTRAINT CHECKS
# ──────────────────────────────────────────────────────────────

print("\n")
print("=" * 72)
print("PHYSICAL CONSTRAINT CHECKS")
print("=" * 72)

constraint_checks = {

    "gas_cf":
        (0, 100),

    "solar_cf":
        (0, 35),

    "eaf":
        (0, 100),

    "Gas_constraint":
        (0, None),

    "Distribution_constraint":
        (0, None),

    "Transmission_constraint":
        (0, None),

    "domestic_alloc":
        (0, None),

    "gas_to_power":
        (0, None)
}

for var, limits in constraint_checks.items():

    if var not in panel.columns:
        continue

    lower, upper = limits

    violations = 0

    if lower is not None:

        violations += (
            panel[var] < lower
        ).sum()

    if upper is not None:

        violations += (
            panel[var] > upper
        ).sum()

    print(
        f"{var:<30}"
        f"violations = {violations}"
    )

# ──────────────────────────────────────────────────────────────
# COMPLETENESS REPORT
# ──────────────────────────────────────────────────────────────

report = pd.DataFrame(
    report_rows
)

report["final_completeness_pct"] = [

    round(
        panel[c].notna().mean() * 100,
        1
    )

    if c in panel.columns

    else 0

    for c in report["variable"]
]

report["improvement_pct"] = (

    report["final_completeness_pct"]

    -

    report["original_completeness_pct"]
)

report = report.sort_values(
    "improvement_pct",
    ascending=False
)

print("\n")
print("=" * 72)
print("TOP RECONSTRUCTION IMPROVEMENTS")
print("=" * 72)

print(
    report.head(15)
)
# ──────────────────────────────────────────────────────────────
# FINAL INTEGER ENFORCEMENT
# ──────────────────────────────────────────────────────────────

for col in [
    "freq_collapse",
    "pipeline_down"
]:

    if col not in panel.columns:
        continue

    panel[col] = (

        panel[col]

        .ffill()

        .bfill()

        .fillna(0)

        .round()

        .clip(lower=0)

        .astype(int)
    )

# ──────────────────────────────────────────────────────────────
# SAVE OUTPUTS
# ──────────────────────────────────────────────────────────────

panel.to_csv(
    "master_panel_reconstructed_v2.csv"
)

report.to_csv(
    "reconstruction_report_v2.csv",
    index=False
)

audit_df.to_csv(
    "economic_audit_report.csv",
    index=False
)

# ──────────────────────────────────────────────────────────────
# SUMMARY
# ──────────────────────────────────────────────────────────────

print("\n")
print("=" * 72)
print("RECONSTRUCTION COMPLETE")
print("=" * 72)

print(
    f"Rows: {panel.shape[0]}"
)

print(
    f"Columns: {panel.shape[1]}"
)

print(
    f"Variables >=95% complete: "
    f"{(report['final_completeness_pct'] >= 95).sum()}"
)

print(
    "\nSaved files:"
)

print(
    "  master_panel_reconstructed_v2.csv"
)

print(
    "  reconstruction_report_v2.csv"
)

print(
    "  economic_audit_report.csv"
)

print("\nCELL 15B COMPLETE")



ECONOMIC AUDITS

AUDIT RESULTS
             Variable_1                Variable_2  Expected  Correlation  \
0         vandalism_idx             pipeline_down  positive       0.4759   
1          gas_to_power                    gas_cf  positive       0.1939   
2    Billing_efficiency                     ATC&C  negative      -0.6588   
3         disco_collect           collection_loss  negative      -0.8878   
4  Total_E_Rec_by_disco  Transmission_Loss_factor  negative      -0.1794   

  Status  
0   PASS  
1   PASS  
2   PASS  
3   PASS  
4   PASS  


PHYSICAL CONSTRAINT CHECKS
gas_cf                        violations = 0
solar_cf                      violations = 0
eaf                           violations = 0
Gas_constraint                violations = 0
Distribution_constraint       violations = 0
Transmission_constraint       violations = 0
domestic_alloc                violations = 0
gas_to_power                  violations = 0


TOP RECONSTRUCTION IMPROVEMENTS
                   va

In [5]:
for col in [
    "market_shortfall",
    "Transmission_constraint",
    "Distribution_constraint"
]:
    print(
        col,
        panel[col].isna().sum()
    )

market_shortfall 0
Transmission_constraint 0
Distribution_constraint 0


In [6]:
panel.isna().sum().sort_values(
    ascending=False
).head(20)

precip_mm         2
brent             0
ttf_jkm           0
jkm_netback       0
ghi               0
usdngn            0
dni               0
temp_c            0
aerosol_od        0
fx_reserves       0
inflation         0
gas_price         0
harmattan         0
fm_dummy          0
pipeline_down     0
vandalism_idx     0
plant_age_gas     0
gas_to_power      0
domestic_alloc    0
eaf               0
dtype: int64